<a href="https://colab.research.google.com/github/cpdong/public/blob/master/test/LLMBind_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧬 LLMBind

**De novo protein binder design from a target.**

An LLM generates candidate binder sequences for a target, which are then filtered by a PPI
classifier (*bindscan*) and, optionally, a structure filter (*NetSurfP-3.0*).

**Runtime:** this is a PyTorch/CUDA pipeline — set **Runtime → Change runtime type → GPU (T4)**.
_(No TPU / JAX / TensorFlow is used.)_

**Two steps:**
1. **Install** (~4 min, run once) — also fetches the demo target (`PDL1.fasta`) and pre-downloads
   the generation model from HuggingFace (`cpdong/test`).
2. **Generate binders** — uses the bundled `PDL1.fasta` target by default. Nothing is fetched
   from GitHub or HuggingFace at run time.


In [ ]:
#@title 1. Install LLMBind  &  fetch demo + model (~4 min){ display-mode: "form" }
#@markdown Run this **once**. Installs the PyTorch-based pipeline
#@markdown (`transformers` + `fair-esm` + NetSurfP-3.0), fetches the demo target
#@markdown (`PDL1.fasta`), and pre-downloads the generation model from HuggingFace.
#@markdown No TPU / JAX / TensorFlow needed — uses Colab's built-in CUDA PyTorch.

#@markdown ---
#@markdown Generation model on the Hub. Only needed here if it is **private/gated** —
#@markdown leave blank for public repos.
hf_model_repo = "cpdong/test"  #@param {type:"string"}
hf_token      = ""             #@param {type:"string"}

import os, time, subprocess

t0 = time.time()
WORK_DIR = "/content/LLMBind_demo"
LLM_DIR  = os.path.join(WORK_DIR, "llm_model")
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

def sh(cmd):
    print(">>", cmd)
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise RuntimeError("command failed: " + cmd)

if not os.path.isfile(os.path.join(WORK_DIR, "READY")):
    # Python deps. torch / torchvision ship with Colab (CUDA build) — don't reinstall them.
    sh("pip -q install "
       "transformers==4.57.3 accelerate fair-esm 'huggingface_hub>=0.23' "
       "'numpy==1.26.4' scipy==1.13.1 scikit-learn==1.6.1 "
       "pandas h5py 'pyyaml==6.0.2' 'requests==2.32.3' biopython matplotlib")

    # NetSurfP-3.0 (structure filter) — a torch-based fork, no TensorFlow
    sh("pip -q install git+https://github.com/cpdong/NetSurfP_3.0_standalone.git")

    # Demo + PPI model files (target is the PDL1.fasta sequence)
    BASE = "https://raw.githubusercontent.com/cpdong/public/refs/heads/master/test"
    for f in ["run_test.py", "bs_model.pt", "PDL1.fasta"]:
        sh(f"wget -q {BASE}/{f} -O {f}")

    # Pre-download the generation model from the Hub into LLM_DIR (so step 2 hits no network)
    print(f"\nDownloading generation model '{hf_model_repo}' -> {LLM_DIR}")
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(
            repo_id=hf_model_repo,
            local_dir=LLM_DIR,
            token=(hf_token or None),
        )
        print("Model downloaded.")
    except Exception as e:
        print(
            f"\n⚠️  Could not download '{hf_model_repo}': {e}\n"
            "    If the repo is private/gated, paste an access token in `hf_token` above\n"
            "    (https://huggingface.co/settings/tokens) and re-run this cell."
        )
        raise

    # Preload ESM weights into the torch hub cache
    sh('python -c "import esm; esm.pretrained.esm1b_t33_650M_UR50S()"')  # NetSurfP-3.0
    sh('python -c "import esm; esm.pretrained.esm2_t6_8M_UR50D()"')      # bindscan PPI

    open(os.path.join(WORK_DIR, "READY"), "w").close()
    print("\n✅ LLMBind setup complete.")
else:
    print("Already installed (delete /content/LLMBind_demo/READY to reinstall).")

# numpy was pinned to <2; if it was downgraded after import you may need to
# Runtime → Restart session once, then continue at step 2 (no need to reinstall).
print(f"\nElapsed: {time.time() - t0:.0f}s")


In [ ]:
#@title 2. Generate binders{ display-mode: "form" }

#@markdown ### Target
#@markdown Uses the bundled **PDL1.fasta** target fetched in step 1 — no network access at run time.
target_fasta = "/content/LLMBind_demo/PDL1.fasta"  #@param {type:"string"}

#@markdown ### Generation model
#@markdown Pre-downloaded from HuggingFace in step 1. Leave as-is to use it.
llm_model = "/content/LLMBind_demo/llm_model"  #@param {type:"string"}

#@markdown ### Design options
num_designs         = 100  #@param {type:"integer"}
generate_batch_size = 16   #@param {type:"integer"}
min_length          = 50   #@param {type:"integer"}
max_length          = 130  #@param {type:"integer"}

#@markdown ### PPI (bindscan) filter
ppi_threshold = 0.3  #@param {type:"number"}

#@markdown ### Structure filter (NetSurfP-3.0) — optional
#@markdown Turn on and give an `nsp3.pth` weights file to enable it. ESM-1b weights are
#@markdown already cached from step 1. Leave off to skip structure filtering.
enable_structure_filter = False  #@param {type:"boolean"}
nsp3_model              = ""     #@param {type:"string"}
min_structured_fraction = 0.6    #@param {type:"number"}

output_dir = "/content/llmbind_output"  #@param {type:"string"}

# ---------------------------------------------------------------------------
import os, shlex, subprocess
from pathlib import Path

WORK_DIR = "/content/LLMBind_demo"

if not os.path.isfile(target_fasta):
    raise FileNotFoundError(
        f"Target FASTA not found: {target_fasta}. Run step 1 first."
    )
if not os.path.isdir(llm_model) or not os.listdir(llm_model):
    raise FileNotFoundError(
        f"Generation model not found at {llm_model}. Run step 1 to download it."
    )

Path(output_dir).mkdir(parents=True, exist_ok=True)

cmd = [
    "python", os.path.join(WORK_DIR, "run_test.py"),
    "--target_fasta",        target_fasta,
    "--gen_model",           llm_model,
    "--ppi_model",           os.path.join(WORK_DIR, "bs_model.pt"),
    "--ppi_threshold",       str(ppi_threshold),
    "--num_designs",         str(num_designs),
    "--generate_batch_size", str(generate_batch_size),
    "--min_length",          str(min_length),
    "--max_length",          str(max_length),
    "--output_dir",          output_dir,
]

if enable_structure_filter and nsp3_model:
    cmd += ["--nsp3_model", nsp3_model,
            "--min_structured_fraction", str(min_structured_fraction)]
else:
    cmd += ["--no_structure_filter"]

print("Running:\n  " + " ".join(shlex.quote(x) for x in cmd) + "\n")
subprocess.run(cmd, check=True)

print("\nFinished. Output files:")
for p in sorted(Path(output_dir).rglob("*")):
    if p.is_file():
        print("  ", p)
